# Khám phá dữ liệu (EDA) và Phân chia tập dữ liệu CASIA-Iris-Thousand

Notebook này thực hiện giai đoạn khảo sát dữ liệu và thiết lập bộ chia tập chuẩn cho dự án nhận diện mống mắt trên bộ dữ liệu **CASIA-Iris-Thousand**, bao gồm:
- Kiểm tra cấu trúc thư mục, tính toàn vẹn và chất lượng 20.000 ảnh hồng ngoại gần (NIR).
- Trực quan hóa phân bố số lượng mẫu theo lớp mắt, các chỉ số chất lượng ảnh và các dạng nhiễu đặc trưng.
- Tạo bộ chia tập cố định cho hai kịch bản **Nhận dạng đóng (`1:N` Closed-set)** và **Xác thực mở (`1:1` Open-set)** kèm kiểm tra chống rò rỉ dữ liệu (data leakage).

## 1. Thiết lập môi trường và nạp cấu hình

Kết nối Google Drive (khi chạy trên Google Colab), đồng bộ mã nguồn dự án và nạp tệp cấu hình `configs/dataset_config.yaml`.

In [ ]:
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Badan174/Project3-Iris.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/prj3")
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    WORK_DIR = Path("/content/Project3-Iris")

    if not WORK_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(WORK_DIR), "pull"], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"],
        check=True,
    )
    DATA_ROOT = Path("/content/data")
else:
    DRIVE_PROJECT_DIR = None
    WORK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_ROOT = WORK_DIR / "data" / "raw"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print("Môi trường Colab :", IN_COLAB)
print("Thư mục mã nguồn :", WORK_DIR)
print("Thư mục dữ liệu  :", DATA_ROOT)
print("Thư mục Drive    :", DRIVE_PROJECT_DIR)

In [ ]:
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import yaml
from PIL import Image

from src.data import (
    build_dataset_summary,
    create_closed_set_splits,
    create_open_set_splits,
    generate_verification_pairs,
    scan_casia_thousand,
    validate_images,
    verify_no_leakage,
)

with open(WORK_DIR / "configs" / "dataset_config.yaml", "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

print("Bộ dữ liệu:", CONFIG["dataset_name"], "| Seed:", CONFIG["seed"])

## 2. Giải nén dữ liệu CASIA-Iris-Thousand

Tệp nén lưu trữ trên Google Drive được giải nén ra ổ đĩa cục bộ (`/content/data`) trước khi đọc hàng loạt nhằm đảm bảo hiệu năng I/O.

In [ ]:
DRIVE_ZIP_PATH = (
    DRIVE_PROJECT_DIR / "CASIA-Iris-Thousand.zip"
    if DRIVE_PROJECT_DIR is not None
    else WORK_DIR / "data" / "CASIA-Iris-Thousand.zip"
)
EXTRACT_TARGET = DATA_ROOT / "CASIA-Iris-Thousand"


def resolve_casia_root(base_dir: Path) -> Path:
    """Xác định thư mục gốc chứa các thư mục đối tượng ('000', '001', ...)."""
    if (base_dir / "000").is_dir():
        return base_dir
    for candidate in base_dir.rglob("000"):
        if candidate.is_dir():
            return candidate.parent
    return base_dir


if not EXTRACT_TARGET.exists() or not any(EXTRACT_TARGET.iterdir()):
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(f"Không tìm thấy tệp nén dữ liệu tại {DRIVE_ZIP_PATH}.")
    EXTRACT_TARGET.mkdir(parents=True, exist_ok=True)
    print(f"Giải nén {DRIVE_ZIP_PATH} -> {EXTRACT_TARGET}")
    with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_TARGET)

RAW_DATA_DIR = resolve_casia_root(EXTRACT_TARGET)
print("Thư mục gốc dữ liệu:", RAW_DATA_DIR)
sample_subdirs = sorted([p.name for p in RAW_DATA_DIR.iterdir() if p.is_dir()])[:10]
print("10 thư mục đối tượng đầu tiên:", sample_subdirs)

## 3. Khảo sát cấu trúc và chất lượng ảnh

Quét toàn bộ tập ảnh để trích xuất siêu dữ liệu (`subject_id`, `eye_side`, `eye_id`), kiểm tra tính hợp lệ của tệp ảnh và tính các chỉ số chất lượng gồm:
- `mean_intensity`: cường độ sáng trung bình trên thang xám (0–255).
- `specular_ratio`: tỷ lệ điểm ảnh có cường độ `>= 245`, phản ánh mức độ phản xạ ánh sáng trên giác mạc hoặc tròng kính.
- `laplacian_var`: phương sai toán tử Laplacian, đo độ sắc nét của vùng ảnh.

In [ ]:
extensions = tuple(CONFIG["image_extensions"])
specular_thresh = int(CONFIG["quality_thresholds"]["specular_pixel_threshold"])

scanned_df = scan_casia_thousand(RAW_DATA_DIR, extensions=extensions)
meta_df = validate_images(
    scanned_df,
    RAW_DATA_DIR,
    check_pixel_stats=True,
    specular_threshold=specular_thresh,
    show_progress=True,
)

valid_df = meta_df[meta_df["is_valid"]].copy()
invalid_df = meta_df[~meta_df["is_valid"]].copy()

print(f"Tổng số tệp ảnh quét được : {len(meta_df):,}")
print(f"Số ảnh hợp lệ              : {len(valid_df):,}")
print(f"Số ảnh lỗi                 : {len(invalid_df):,}")
print(f"Số đối tượng (subjects)    : {valid_df['subject_id'].nunique():,}")
print(f"Số lớp mắt (L và R riêng)  : {valid_df['eye_id'].nunique():,}")
print(
    "Phân bố độ phân giải       :",
    (valid_df["width"].astype(str) + "x" + valid_df["height"].astype(str))
    .value_counts()
    .to_dict(),
)

display_cols = ["mean_intensity", "std_intensity", "specular_ratio", "laplacian_var"]
valid_df[display_cols].describe().round(4)

## 4. Trực quan hóa dữ liệu

Thực hiện 4 nhóm biểu đồ phân tích và lưu kết quả vào `results/eda/`:
1. Cân bằng số lượng ảnh trên mỗi lớp mắt và theo hai bên mắt (`L` / `R`).
2. Phân bố các chỉ số chất lượng (`mean_intensity`, `specular_ratio`, `laplacian_var`).
3. Đối chiếu ảnh mắt trái (`L`) và mắt phải (`R`) trên cùng một đối tượng nhằm minh họa tính độc lập về cấu trúc vân mống mắt giữa hai mắt.
4. Các mẫu nhiễu điển hình (tỷ lệ phản xạ sáng cao nhất do đeo kính/chói đèn và độ nét thấp nhất do mờ hoặc che khuất).

In [ ]:
EDA_DIR = WORK_DIR / "results" / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

# Biểu đồ 1: Cân bằng số lượng ảnh theo lớp mắt và bên mắt (L / R)
eye_counts = valid_df.groupby("eye_id").size()
side_counts = valid_df["eye_side"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
eye_counts.value_counts().sort_index().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Phân bố số lượng ảnh trên mỗi lớp mắt")
axes[0].set_xlabel("Số ảnh mỗi mắt")
axes[0].set_ylabel("Số lớp mắt")
axes[0].grid(axis="y", alpha=0.3)

side_counts.plot(kind="bar", ax=axes[1], color=["teal", "coral"])
axes[1].set_title("Tổng số ảnh theo bên mắt (L và R)")
axes[1].set_xlabel("Bên mắt")
axes[1].set_ylabel("Số ảnh")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(EDA_DIR / "images_per_eye.png", dpi=150, bbox_inches="tight")
plt.show()

# Biểu đồ 2: Phân bố 3 chỉ số chất lượng ảnh
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(valid_df["mean_intensity"], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_title("Phân bố độ sáng trung bình (Mean Intensity)")
axes[0].set_xlabel("Intensity (0-255)")

axes[1].hist(valid_df["specular_ratio"] * 100.0, bins=40, color="#DD8452", edgecolor="white")
axes[1].set_title("Tỷ lệ điểm sáng phản xạ (>= 245)")
axes[1].set_xlabel("Specular Ratio (%)")

axes[2].hist(valid_df["laplacian_var"], bins=40, color="#55A868", edgecolor="white")
axes[2].set_title("Phân bố độ nét (Laplacian Variance)")
axes[2].set_xlabel("Laplacian Var")

for ax in axes:
    ax.set_ylabel("Số ảnh")
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(EDA_DIR / "quality_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Biểu đồ 3: Đối chiếu 5 mẫu mắt trái (L) và 5 mẫu mắt phải (R) của cùng một đối tượng
sample_subject = sorted(valid_df["subject_id"].unique())[0]
left_rows = valid_df[(valid_df["subject_id"] == sample_subject) & (valid_df["eye_side"] == "L")].head(5)
right_rows = valid_df[(valid_df["subject_id"] == sample_subject) & (valid_df["eye_side"] == "R")].head(5)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for col_idx, (_, row) in enumerate(left_rows.iterrows()):
    img = Image.open(RAW_DATA_DIR / row["rel_path"])
    axes[0, col_idx].imshow(img, cmap="gray")
    axes[0, col_idx].set_title(f"{row['filename']} (L)", fontsize=9)
    axes[0, col_idx].axis("off")

for col_idx, (_, row) in enumerate(right_rows.iterrows()):
    img = Image.open(RAW_DATA_DIR / row["rel_path"])
    axes[1, col_idx].imshow(img, cmap="gray")
    axes[1, col_idx].set_title(f"{row['filename']} (R)", fontsize=9)
    axes[1, col_idx].axis("off")

fig.suptitle(f"Đối tượng {sample_subject}: Mắt trái (hàng trên) và Mắt phải (hàng dưới)", fontsize=13)
plt.tight_layout()
plt.savefig(EDA_DIR / "sample_left_vs_right.png", dpi=150, bbox_inches="tight")
plt.show()

# Biểu đồ 4: Top 8 ảnh phản xạ sáng cao nhất và Top 8 ảnh có độ nét thấp nhất
top_glare = valid_df.nlargest(8, "specular_ratio")
lowest_sharp = valid_df.nsmallest(8, "laplacian_var")

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for col_idx, (_, row) in enumerate(top_glare.iterrows()):
    img = Image.open(RAW_DATA_DIR / row["rel_path"])
    axes[0, col_idx].imshow(img, cmap="gray")
    axes[0, col_idx].set_title(f"Glare: {row['specular_ratio']*100:.1f}%", fontsize=8)
    axes[0, col_idx].axis("off")

for col_idx, (_, row) in enumerate(lowest_sharp.iterrows()):
    img = Image.open(RAW_DATA_DIR / row["rel_path"])
    axes[1, col_idx].imshow(img, cmap="gray")
    axes[1, col_idx].set_title(f"LapVar: {row['laplacian_var']:.1f}", fontsize=8)
    axes[1, col_idx].axis("off")

fig.suptitle("Các mẫu nhiễu điển hình: Hàng trên = Phản xạ sáng/Kính | Hàng dưới = Mờ/Che khuất", fontsize=12)
plt.tight_layout()
plt.savefig(EDA_DIR / "hard_noisy_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Phân chia tập dữ liệu (1:N & 1:1) và kiểm tra chống rò rỉ dữ liệu

Thiết lập bộ chia dùng chung cho cả phương pháp truyền thống và học sâu:
- **Closed-set (`1:N`)**: Phân tầng theo từng lớp mắt với tỷ lệ `6 Train / 2 Val / 2 Test`, kèm cờ `in_subset_200_eyes` cho tập con 100 đối tượng đầu (200 lớp mắt).
- **Open-set (`1:1`)**: Tách biệt hoàn toàn theo đối tượng (`700 Train / 150 Val / 150 Test`) và sinh danh sách cặp đối sánh `val_pairs.csv`, `test_pairs.csv` (trong đó 25% cặp mạo danh là mắt trái và mắt phải của cùng một người).
- **Kiểm tra toàn vẹn**: Xác minh không có sự trùng lặp ảnh, lớp mắt hay đối tượng giữa các tập thông qua `verify_no_leakage`, đồng thời xuất các tệp kết quả vào `data/splits/` và đồng bộ lưu trữ sang Google Drive (nếu chạy trên Colab).

In [ ]:
seed = int(CONFIG["seed"])
closed_cfg = CONFIG["closed_set"]
open_cfg = CONFIG["open_set"]
pairs_cfg = CONFIG["verification_pairs"]

closed_df = create_closed_set_splits(
    valid_df,
    train_per_eye=int(closed_cfg["train_per_eye"]),
    val_per_eye=int(closed_cfg["val_per_eye"]),
    test_per_eye=int(closed_cfg["test_per_eye"]),
    subset_num_subjects=int(closed_cfg["subset_num_subjects"]),
    seed=seed,
)

open_df = create_open_set_splits(
    valid_df,
    train_subjects=int(open_cfg["train_subjects"]),
    val_subjects=int(open_cfg["val_subjects"]),
    test_subjects=int(open_cfg["test_subjects"]),
    seed=seed,
)

val_pairs = generate_verification_pairs(
    open_df[open_df["split"] == "val"],
    num_genuine=int(pairs_cfg["val_num_genuine"]),
    num_impostor=int(pairs_cfg["val_num_impostor"]),
    same_subject_impostor_ratio=float(pairs_cfg["same_subject_impostor_ratio"]),
    seed=seed + 1,
)

test_pairs = generate_verification_pairs(
    open_df[open_df["split"] == "test"],
    num_genuine=int(pairs_cfg["test_num_genuine"]),
    num_impostor=int(pairs_cfg["test_num_impostor"]),
    same_subject_impostor_ratio=float(pairs_cfg["same_subject_impostor_ratio"]),
    seed=seed + 2,
)

leakage_report = verify_no_leakage(closed_df, open_df, val_pairs, test_pairs)

print("=== CLOSED-SET (1:N) ===")
print(closed_df.groupby(["split", "in_subset_200_eyes"]).size())
print("\n=== OPEN-SET (1:1) ===")
print(
    open_df.groupby("split").agg(
        images=("rel_path", "count"),
        subjects=("subject_id", "nunique"),
        eyes=("eye_id", "nunique"),
    )
)
print("\n=== CẶP ĐỐI SÁNH (VERIFICATION PAIRS) ===")
print("Tập Val :", val_pairs["pair_type"].value_counts().to_dict())
print("Tập Test:", test_pairs["pair_type"].value_counts().to_dict())
print("\nKiểm tra chống rò rỉ dữ liệu đạt chuẩn:", leakage_report["passed"])

In [ ]:
SPLITS_DIR = WORK_DIR / "data" / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

meta_df.to_csv(SPLITS_DIR / "metadata.csv", index=False)
closed_df.to_csv(SPLITS_DIR / "closed_set_splits.csv", index=False)
open_df.to_csv(SPLITS_DIR / "open_set_splits.csv", index=False)
val_pairs.to_csv(SPLITS_DIR / "val_pairs.csv", index=False)
test_pairs.to_csv(SPLITS_DIR / "test_pairs.csv", index=False)

summary = build_dataset_summary(
    meta_df, closed_df, open_df, val_pairs, test_pairs, leakage_report, CONFIG
)
with open(SPLITS_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

if DRIVE_PROJECT_DIR is not None:
    drive_splits = DRIVE_PROJECT_DIR / "splits"
    drive_eda = DRIVE_PROJECT_DIR / "eda"
    drive_splits.mkdir(parents=True, exist_ok=True)
    drive_eda.mkdir(parents=True, exist_ok=True)

    for src_file in SPLITS_DIR.glob("*"):
        if src_file.is_file() and src_file.name != ".gitkeep":
            shutil.copy2(src_file, drive_splits / src_file.name)
    for src_file in EDA_DIR.glob("*.png"):
        shutil.copy2(src_file, drive_eda / src_file.name)

    print(f"Đã đồng bộ tệp splits -> {drive_splits}")
    print(f"Đã đồng bộ biểu đồ EDA -> {drive_eda}")

print("Hoàn tất xuất bộ chia tập tại:", SPLITS_DIR)